In [ ]:
# === DINOv2 ViT-L/14 eval run ===
# Prerequisite: repo is cloned at /content/Bricky and the eval set already exists
# at /content/Bricky/Tools/dinov2-embeddings/eval (from the earlier build step).
# If you're in a fresh runtime, the two lines at the top handle that.

import os
REPO = "/content/Bricky"
if not os.path.exists(REPO):
    !git clone https://github.com/shribr/Bricky.git /content/Bricky

%cd /content/Bricky
!pip install -q -r Tools/dinov2-embeddings/requirements.txt

# Mount Drive for persistent report output.
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
!mkdir -p /content/drive/MyDrive/Bricky/reports

# Build the eval set if it isn't there already (deterministic given the seed).
if not os.path.exists("Tools/dinov2-embeddings/eval/ground_truth.json"):
    !python Tools/dinov2-embeddings/build_eval_set.py --count 200 --variants-per-figure 8 --seed 42

# Apply the bbox-crop patch if not already applied (idempotent).
from pathlib import Path
p = Path("Tools/dinov2-embeddings/evaluate_retrieval.py")
txt = p.read_text()
if "detect_figure_bbox" not in txt:
    FN = '''

def detect_figure_bbox(img):
    import numpy as np
    arr = np.asarray(img.convert("RGB"), dtype=np.float32)
    corners = np.concatenate([arr[0:5].reshape(-1,3), arr[-5:].reshape(-1,3),
                              arr[:,0:5].reshape(-1,3), arr[:,-5:].reshape(-1,3)])
    bg = corners.mean(axis=0)
    dist = np.sqrt(((arr - bg) ** 2).sum(axis=2))
    ys, xs = np.where(dist > 40)
    if len(xs) == 0:
        return (0, 0, img.width, img.height)
    pad = 4
    return (max(0,int(xs.min())-pad), max(0,int(ys.min())-pad),
            min(img.width,int(xs.max())+pad), min(img.height,int(ys.max())+pad))
'''
    txt = txt.replace("K_RECALL = [1, 5, 10, 50]", "K_RECALL = [1, 5, 10, 50]" + FN, 1)
    txt = txt.replace(
        '        img = Image.open(p).convert("RGB")\n        crop = torso_crop(img)',
        '        img = Image.open(p).convert("RGB")\n'
        '        img = img.crop(detect_figure_bbox(img))\n'
        '        crop = torso_crop(img)', 1)
    p.write_text(txt)
    print("evaluator patched")

# Embed the catalog with ViT-L/14.
!python Tools/dinov2-embeddings/embed_catalog.py \
    --model dinov2_vitl14 \
    --out Tools/dinov2-embeddings/index/dinov2_vitl14

# Score.
REPORT = "/content/drive/MyDrive/Bricky/reports/dinov2_vitl14.json"
!python Tools/dinov2-embeddings/evaluate_retrieval.py \
    --index Tools/dinov2-embeddings/index/dinov2_vitl14 \
    --report {REPORT}

# Dump the report so you can copy it straight out of the cell output.
print("\n\n=============== REPORT ({}) ===============".format(REPORT))
!cat {REPORT}